## Getting the Data


In [59]:
import pandas as pd
import re

In [32]:
messages = pd.read_csv(
    "data/SMSSpamCollection", delimiter="\t", names=["label", "message"]
)

In [33]:
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Data Cleaning and Preprocessing


In [34]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [35]:
porterstemmer = PorterStemmer()

In [36]:
corpus = []

for i in range(len(messages)):
    message = re.sub("[^a-zA-Z]", " ", messages["message"][i])
    message = message.lower()
    message = message.split()

    message = [
        porterstemmer.stem(word)
        for word in message
        if word not in set(stopwords.words("english"))
    ]

    message = " ".join(message)

    corpus.append(message)

## Apply BOWs


In [37]:
from sklearn.feature_extraction.text import CountVectorizer

In [38]:
countvectorizer = CountVectorizer(
    max_features=100, binary=True
)  # take the 2500 most frequent words

In [39]:
X = countvectorizer.fit_transform(corpus).toarray()

In [40]:
X.shape  # columns are 2500 because we have taken the 2500 most frequent words

(5572, 100)

In [41]:
countvectorizer.vocabulary_  # shows the words and their index in the array

{'go': np.int64(22),
 'great': np.int64(25),
 'got': np.int64(24),
 'wat': np.int64(90),
 'ok': np.int64(56),
 'free': np.int64(18),
 'win': np.int64(94),
 'text': np.int64(77),
 'txt': np.int64(85),
 'say': np.int64(67),
 'alreadi': np.int64(0),
 'think': np.int64(80),
 'hey': np.int64(28),
 'week': np.int64(92),
 'back': np.int64(3),
 'like': np.int64(38),
 'still': np.int64(73),
 'send': np.int64(69),
 'even': np.int64(15),
 'friend': np.int64(19),
 'prize': np.int64(62),
 'claim': np.int64(7),
 'call': np.int64(4),
 'mobil': np.int64(47),
 'co': np.int64(8),
 'home': np.int64(30),
 'want': np.int64(89),
 'today': np.int64(82),
 'cash': np.int64(6),
 'day': np.int64(12),
 'repli': np.int64(64),
 'www': np.int64(96),
 'right': np.int64(65),
 'thank': np.int64(78),
 'take': np.int64(75),
 'time': np.int64(81),
 'use': np.int64(87),
 'messag': np.int64(44),
 'oh': np.int64(55),
 'ye': np.int64(97),
 'make': np.int64(42),
 'way': np.int64(91),
 'feel': np.int64(16),
 'dont': np.int64(14

# BOW with N Grams


In [42]:
countvectorizer = CountVectorizer(max_features=100, binary=True, ngram_range=(2, 2))

X = countvectorizer.fit_transform(corpus).toarray()

countvectorizer.vocabulary_  # shows the words and their index in the array

{'free entri': np.int64(30),
 'claim call': np.int64(15),
 'call claim': np.int64(3),
 'free call': np.int64(29),
 'chanc win': np.int64(14),
 'txt word': np.int64(90),
 'let know': np.int64(53),
 'go home': np.int64(34),
 'pleas call': np.int64(67),
 'lt gt': np.int64(57),
 'want go': np.int64(97),
 'like lt': np.int64(54),
 'sorri call': np.int64(81),
 'call later': np.int64(8),
 'ur award': np.int64(91),
 'call custom': np.int64(4),
 'custom servic': np.int64(22),
 'cash prize': np.int64(13),
 'po box': np.int64(68),
 'tri contact': np.int64(87),
 'draw show': np.int64(27),
 'show prize': np.int64(79),
 'prize guarante': np.int64(73),
 'guarante call': np.int64(42),
 'valid hr': np.int64(95),
 'select receiv': np.int64(76),
 'privat account': np.int64(71),
 'account statement': np.int64(0),
 'statement show': np.int64(83),
 'call identifi': np.int64(5),
 'identifi code': np.int64(48),
 'code expir': np.int64(19),
 'urgent mobil': np.int64(94),
 'call landlin': np.int64(7),
 'wat tim

## Encoding Target Variable (y)


In [51]:
# Encoding dependent variable: ham -> 0, spam -> 1
y = pd.get_dummies(messages["label"], drop_first=True)
y = y.iloc[:, 0].values

## Train Test Split


In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=0
)

## Training Model using Naive Bayes Classifier


In [53]:
from sklearn.naive_bayes import MultinomialNB

spam_detect_model = MultinomialNB().fit(X_train, y_train)

## Model Prediction


In [54]:
y_pred = spam_detect_model.predict(X_test)

## Model Evaluation


In [55]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [56]:
confusion_m = confusion_matrix(y_test, y_pred)
confusion_m

array([[954,   1],
       [ 71,  89]])

In [57]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.9354260089686098

In [58]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.93      1.00      0.96       955
        True       0.99      0.56      0.71       160

    accuracy                           0.94      1115
   macro avg       0.96      0.78      0.84      1115
weighted avg       0.94      0.94      0.93      1115

